In [4]:
# ===== IMPORTS =====
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np

# ===== LOAD DATASET =====
data = fetch_california_housing(as_frame=True)
df = data.frame

# Focus on MedInc
population = df['MedInc']

# True population mean
true_mean = population.mean()
print("True Population Mean:", round(true_mean, 4))

# ===== PART (i) - INTRODUCE MISSING VALUES =====
np.random.seed(42)

df_missing = df.copy()

# Each row has 20% chance of becoming missing
missing_mask = np.random.rand(len(df_missing)) < 0.2
df_missing.loc[missing_mask, 'MedInc'] = np.nan

print("Missing proportion:", round(df_missing['MedInc'].isna().mean(), 4))
print("Total missing values:", df_missing['MedInc'].isna().sum())

"""df_missing['MedInc']
→ Selects the MedInc column from our DataFrame
→ This column has some real numbers and some NaN values
→ Example: [3.5, NaN, 4.2, NaN, 2.1, NaN ...]

.isna()
→ Checks every value — is it NaN or not?
→ Returns True for NaN, False for real numbers
"""
# ===== PART (ii) - HANDLE MISSING DATA =====

# Method 1: Listwise Deletion
df_listwise = df_missing.dropna(subset=['MedInc'])
print("\n--- Listwise Deletion ---")
print("Original size :", len(df))
print("After deletion:", len(df_listwise))
print("Rows removed  :", len(df) - len(df_listwise))

# Method 2: Mean Imputation
df_imputed = df_missing.copy()
mean_value = df_imputed['MedInc'].mean()
print("\n--- Mean Imputation ---")
print("Value used for imputation:", round(mean_value, 4))
df_imputed['MedInc'] = df_imputed['MedInc'].fillna(mean_value)
print("Missing after imputation:", df_imputed['MedInc'].isna().sum())

# ===== SAMPLING FUNCTIONS =====
def srswr(series, n):
    sample = series.sample(n=n, replace=True)
    return sample.mean()

def srswor(series, n):
    sample = series.sample(n=n, replace=False)
    return sample.mean()

# ===== PART (iii) - COMPARE SRSWR AND SRSWOR =====
n = 500

# Original data
mean_wr_original  = srswr(population, n)
mean_wor_original = srswor(population, n)

# Listwise deletion
mean_wr_listwise  = srswr(df_listwise['MedInc'], n)
mean_wor_listwise = srswor(df_listwise['MedInc'], n)

# Mean imputation
mean_wr_imputed  = srswr(df_imputed['MedInc'], n)
mean_wor_imputed = srswor(df_imputed['MedInc'], n)

# ===== PRINT RESULTS =====
print("\n========== ESTIMATES ==========")
print(f"True Population Mean : {round(true_mean, 4)}")

print("\nOriginal Data (No Missing):")
print("  SRSWR :", round(mean_wr_original, 4))
print("  SRSWOR:", round(mean_wor_original, 4))

print("\nListwise Deletion:")
print("  SRSWR :", round(mean_wr_listwise, 4))
print("  SRSWOR:", round(mean_wor_listwise, 4))

print("\nMean Imputation:")
print("  SRSWR :", round(mean_wr_imputed, 4))
print("  SRSWOR:", round(mean_wor_imputed, 4))

# ===== BIAS CALCULATION =====
print("\n========== BIAS (difference from true mean) ==========")
print(f"Original  SRSWR  bias: {round(abs(mean_wr_original  - true_mean), 4)}")
print(f"Original  SRSWOR bias: {round(abs(mean_wor_original - true_mean), 4)}")
print(f"Listwise  SRSWR  bias: {round(abs(mean_wr_listwise  - true_mean), 4)}")
print(f"Listwise  SRSWOR bias: {round(abs(mean_wor_listwise - true_mean), 4)}")
print(f"Imputed   SRSWR  bias: {round(abs(mean_wr_imputed   - true_mean), 4)}")
print(f"Imputed   SRSWOR bias: {round(abs(mean_wor_imputed  - true_mean), 4)}")


# ===== PART (iv) - COMMENTS =====
print("""
========== COMMENTS ON IMPACT OF MISSING DATA ==========

1. ORIGINAL DATA:
   Both SRSWR and SRSWOR give estimates very
   close to true mean. This is our benchmark.

2. LISTWISE DELETION:
   Removes 20% of rows (around 4000 districts).
   Mean estimate is still close to true mean
   because missing values were random (MCAR).
   Risk: if missing not random, estimates get biased.

3. MEAN IMPUTATION:
   No rows removed - dataset size stays same.
   Mean estimate close to true mean.
   BUT variance is reduced artificially because
   all missing values become the same number.
   This makes the data look less spread out
   than it really is.

4. SRSWR vs SRSWOR:
   SRSWOR is slightly more accurate than SRSWR.
   Because in SRSWOR each unit selected once only.
   In SRSWR same unit can repeat - less efficient.

5. CONCLUSION:
   Best approach = SRSWOR + Listwise Deletion
   (when data is missing completely at random).
   Mean imputation is convenient but hides
   the true spread of the data.
=========================================================
""")

True Population Mean: 3.8707
Missing proportion: 0.1998
Total missing values: 4123

--- Listwise Deletion ---
Original size : 20640
After deletion: 16517
Rows removed  : 4123

--- Mean Imputation ---
Value used for imputation: 3.8858
Missing after imputation: 0

========== ESTIMATES ==========
True Population Mean : 3.8707

Original Data (No Missing):
  SRSWR : 3.9694
  SRSWOR: 3.8715

Listwise Deletion:
  SRSWR : 3.8377
  SRSWOR: 3.7458

Mean Imputation:
  SRSWR : 4.0017
  SRSWOR: 3.8496

========== BIAS (difference from true mean) ==========
Original  SRSWR  bias: 0.0987
Original  SRSWOR bias: 0.0008
Listwise  SRSWR  bias: 0.033
Listwise  SRSWOR bias: 0.1249
Imputed   SRSWR  bias: 0.131
Imputed   SRSWOR bias: 0.0211

========== COMMENTS ON IMPACT OF MISSING DATA ==========

1. ORIGINAL DATA:
   Both SRSWR and SRSWOR give estimates very
   close to true mean. This is our benchmark.

2. LISTWISE DELETION:
   Removes 20% of rows (around 4000 districts).
   Mean estimate is still close t